# 02 — SQL Functions

IrisPark exposes ~120 PySpark-compatible functions across math, string, date/time, conditional, and hash categories, plus IRISPARK-native aggregates. Use `df.explain(extended=True)` to see the execution engine for each function.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Math functions

In [ ]:
from irispark.functions import abs, sqrt, pow, floor, ceil, round as rnd, exp, log, sin, cos, tan, pmod

import pandas as pd

pdf = pd.DataFrame({"x": [1.0, 4.0, 9.0, 16.0], "y": [2.0, 3.0, 4.0, 5.0]})
df = session.createDataFrame(pdf)

df.select(
    "x",
    sqrt("x").alias("sqrt_x"),
    pow("x", 2).alias("x2"),
    floor("x").alias("floor_x"),
    ceil("x").alias("ceil_x"),
    rnd("x", 0).alias("round_x"),
    exp("x").alias("exp_x"),
    log("x").alias("log_x"),
    sin("x").alias("sin_x"),
    pmod("x", 3).alias("pmod_x"),
).show()

## 2. String functions

In [ ]:
from irispark.functions import upper, lower, trim, length, concat, substring, lpad, rpad, initcap, levenshtein, soundex, regexp_extract, split, charindex, find_in_set

pdf = pd.DataFrame({"nome": ["  Ana  ", "Bruno", "carlos", "Diana"], "email": ["ana@x.com", "bruno@y.com", "carlos@z.com", "diana@w.com"]})
df = session.createDataFrame(pdf)

df.select(
    "nome",
    upper("nome").alias("upper"),
    lower("nome").alias("lower"),
    trim("nome").alias("trim"),
    length("nome").alias("len"),
    concat("nome", "email").alias("concat"),
    substring("email", 1, 3).alias("sub"),
    lpad("nome", 10, "*").alias("lpad"),
    initcap("nome").alias("initcap"),
    levenshtein("nome", "nome").alias("lev"),
    regexp_extract("email", "@(.*)", 1).alias("domain"),
    split("email", "@").alias("parts"),
    charindex("@", "email").alias("at_pos"),
).show()

## 3. Date/time functions

In [ ]:
from irispark.functions import year, month, dayofmonth, datediff, date_add, date_sub, add_months, months_between, timestampdiff, current_date, last_day, dayname, monthname

pdf = pd.DataFrame({"d1": ["2025-01-15", "2025-02-01", "2025-03-10"], "d2": ["2025-01-01", "2025-02-28", "2025-03-31"]})
df = session.createDataFrame(pdf)

df.select(
    "d1",
    year("d1").alias("year"),
    month("d1").alias("month"),
    dayofmonth("d1").alias("day"),
    datediff("d2", "d1").alias("diff_days"),
    date_add("d1", 7).alias("plus7"),
    date_sub("d1", 7).alias("minus7"),
    add_months("d1", 1).alias("plus1m"),
    months_between("d2", "d1").alias("months"),
    timestampdiff("DAY", "d1", "d2").alias("ts_diff"),
    last_day("d1").alias("last_day"),
    dayname("d1").alias("dayname"),
    monthname("d1").alias("monthname"),
).show()

print("current_date:", session.sql("SELECT CURRENT_DATE").rows[0][0])

## 4. Conditional functions

In [ ]:
from irispark.functions import when, coalesce, ifnull, nvl, greatest, least, isnull, isnotnull, lit, col

pdf = pd.DataFrame({"a": [1, None, 3, None], "b": [10, 20, None, 40]})
df = session.createDataFrame(pdf)

df.select(
    "a", "b",
    when(col("a").isNull(), lit(0)).otherwise(col("a")).alias("a_or_0"),
    coalesce("a", "b", lit(99)).alias("coalesce"),
    ifnull("a", lit(0)).alias("ifnull"),
    nvl("a", lit(0)).alias("nvl"),
    greatest("a", "b").alias("greatest"),
    least("a", "b").alias("least"),
    isnull("a").alias("isnull_a"),
    isnotnull("a").alias("isnotnull_a"),
).show()

## 5. Hash functions

In [ ]:
from irispark.functions import md5, sha1, sha2, crc32

pdf = pd.DataFrame({"s": ["hello", "world", "irispark"]})
df = session.createDataFrame(pdf)

df.select("s", md5("s").alias("md5"), sha1("s").alias("sha1"), sha2("s", 256).alias("sha256"), crc32("s").alias("crc32")).show()

## 6. IRISPARK-native aggregates

`median`, `percentile`, `quantile`, `skewness`, `kurtosis`, `corr` run inside IRIS (ObjectScript UDAFs or the SQL-native analytic engine).

In [ ]:
from irispark.functions import median, percentile, quantile, skewness, kurtosis, corr

pdf = pd.DataFrame({"x": [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0], "y": [10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0, 1.0]})
df = session.createDataFrame(pdf)

df.select(
    median("x").alias("median"),
    percentile("x", 0.25).alias("p25"),
    quantile("x", 0.9).alias("q90"),
    skewness("x").alias("skew"),
    kurtosis("x").alias("kurt"),
    corr("x", "y").alias("corr"),
).show()

## 7. Execution engine transparency

`df.explain(extended=True)` shows the logical plan, execution mapping, and function registry.

In [ ]:
df.select(median("x").alias("m"), corr("x", "y").alias("c")).explain(extended=True)

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")